# trial-matcher — judge ablation

Judges the same shortlist with several local models, to see which one reads
eligibility criteria best. Nothing here touches the index or the test set: the
shortlist is fixed, produced once by the retrieval stage.

**Before running**, in the panel on the right:

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Input: **Add Input -> Datasets**, the pack built by `src.analysis.pack`, which
  carries the data and the code (its name does not matter, files are found by
  name)

Leave `SMOKE = True` for the first run: one trial, a few minutes, enough to
prove that the models load and the verdicts are written where they should be.

In [ ]:
SMOKE = True

YEAR, RUN = 2021, "dense_medembed-small"

# gemma3 and medgemma at 27B are the same base, size and quantization, so what
# separates them is the medical fine-tune and nothing else. gemma4:12b is the
# judge in use: it answers whether switching is worth it, not why.
MODELS = ["gemma4:12b", "gemma3:27b", "medgemma:27b"]
TOPICS, DEPTH = "12-31", 20

if SMOKE:
    # Topic 1 is a tuning topic: a smoke run gets read line by line, and no
    # topic we report numbers on should be read that way.
    MODELS, TOPICS, DEPTH = ["gemma3:4b"], "1", 1

print(f"{len(MODELS)} models x topics {TOPICS} x top-{DEPTH}")

## 1. Ollama

In [ ]:
import os
import subprocess
import time

import requests

# Not /kaggle/working: the models are tens of gigabytes and everything there is
# saved as the notebook's output.
os.environ["OLLAMA_MODELS"] = "/tmp/ollama"

# zstd: the installer unpacks a zstd archive and the image has none.
# pciutils: without lspci the installer cannot see the GPU, skips the CUDA
# libraries, and ollama then runs on the CPU without saying so.
apt = "apt-get -qq update && apt-get -qq install -y zstd pciutils"
subprocess.run(apt, shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.Popen(
    ["ollama", "serve"],
    start_new_session=True,
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        if requests.get("http://localhost:11434", timeout=2).ok:
            print("ollama is up")
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError(open("/tmp/ollama.log").read())

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
lspci = subprocess.run(["lspci", "-nn"], capture_output=True, text=True).stdout
print(lspci.count("NVIDIA"), "NVIDIA devices on the bus")

## 2. Code and dependencies

The code travels inside the pack. Cloning it here would mean making the
repository public or handing a token to the notebook, and a private clone over
https just sits waiting for a password nobody can type.

In [ ]:
import shutil
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
CODE = Path("/tmp/trial-matcher")


def find(name: str) -> Path:
    """Locate a file of the pack wherever the dataset put it: the slug follows
    the title Kaggle made rather than ours, and an uploaded zip can land one
    folder deeper than it was built."""
    hits = sorted(p for p in INPUT.rglob(name) if p.is_file())
    if not hits:
        raise FileNotFoundError(f"{name} is not under {INPUT}: is the dataset attached?")
    return hits[0]


source = find("judge.py").parent.parent
shutil.rmtree(CODE, ignore_errors=True)
CODE.mkdir(parents=True)
shutil.copytree(source, CODE / "src")
print(f"code: {sum(1 for _ in source.rglob('*.py'))} files from {source}")

!pip install typer tqdm python-dotenv

## 3. Where things are read and written

The pack is mounted read-only, so the verdicts go to `/kaggle/working/out`,
which is what gets saved as this notebook's output. If an earlier session's
verdicts are in the dataset they are brought forward: the cache key is
`(topic, trial, model, prompt)`, so whatever is already judged is skipped
instead of being paid for twice.

In [ ]:
topics_file = find(f"topics{YEAR}.jsonl")
blocks = find("trials_judged.jsonl")
shortlist = find(f"{RUN}{YEAR}.txt")
assert blocks.parent == topics_file.parent, "topics and criteria blocks must sit together"

os.environ["DATA_PROCESSED"] = str(topics_file.parent)
os.environ["RUNS"] = str(WORK / "runs")
os.environ["DATA_OUT"] = str(WORK / "out")
(WORK / "runs").mkdir(exist_ok=True)
(WORK / "out").mkdir(exist_ok=True)
shutil.copy(shortlist, WORK / "runs")

for path in (topics_file, blocks, shortlist):
    print(f"{path.stat().st_size / 1e6:6.1f} MB  {path}")

# Verdicts carried over from an earlier session are skipped instead of redone.
for previous in INPUT.rglob(f"verdicts{YEAR}.jsonl"):
    shutil.copy(previous, WORK / "out")
    print(f"\nresuming from {sum(1 for _ in open(previous))} verdicts in {previous}")
    break

## 4. Judge

One model at a time, using both GPUs. Each model is removed after its turn:
two 27B models are 34 GB and the disk does not hold them together.

In [ ]:
for model in MODELS:
    print(f"\n===== {model} =====", flush=True)
    subprocess.run(["ollama", "pull", model], check=True)

    cmd = ["python", "-m", "src.assess.judge", "--year", str(YEAR), "--run", RUN]
    cmd += ["--topics", TOPICS, "--depth", str(DEPTH), "--model", model]

    started = time.time()
    subprocess.run(cmd, cwd=CODE, check=True)
    print(f"{model}: {(time.time() - started) / 60:.1f} min")

    subprocess.run(["ollama", "rm", model], check=True)

## 5. What came out

In [ ]:
import json
from collections import Counter

path = WORK / "out" / f"verdicts{YEAR}.jsonl"
rows = [json.loads(line) for line in open(path)]
counts = Counter((r["model"], r["prompt_version"]) for r in rows)
for (model, prompt), n in sorted(counts.items()):
    seconds = [r.get("seconds", 0) for r in rows if r["model"] == model]
    mean = sum(seconds) / max(len(seconds), 1)
    print(f"{model:<16} prompt {prompt}  {n:>5} trials  {mean:5.1f} s each")

print(f"\n{path}  ({path.stat().st_size / 1e6:.1f} MB)")

# Where the model actually ran: a CPU fallback is silent and ten times slower.
print()
!grep -iE "offloaded|cuda|rocm" /tmp/ollama.log | tail -5

## 6. Then

**Save Version** to run it to the end without keeping the tab open, then
download `out/verdicts2021.jsonl` from the version's Output. Append it to
`data/processed/verdicts2021.jsonl` in the repo: the rerank runs locally, needs
no model, and compares the judges from those records alone.

```bash
uv run python -m src.assess.rerank --run dense_medembed-small --depth 20 --model medgemma:4b
```

To carry on in a later session, add that same file to a new version of the
dataset: cell 3 picks it up and only the missing trials are judged.